# AI Laboratory: Bayesian Networks and Autoregressive Language Models

**Completed lab record — source:** `BN_lab.pdf`

This notebook implements first- and second-order count-based language models, answers Questions 1–14, tests probability invariants, compares decoding strategies and model complexity, and saves generated sentences.

## Parts I–III: Probability, network assumptions, and data

The chain rule turns sequence generation into repeated conditional prediction. A first-order model uses the directed chain \(X_{t-1}\to X_t\), while a second-order model gives \(X_t\) two parents, \(X_{t-2}\) and \(X_{t-1}\).

In [1]:
from collections import Counter
from itertools import product
from pathlib import Path
import random
import pandas as pd

from ngram_language_models import (
    START, END, tokenize,
    FirstOrderLanguageModel, SecondOrderLanguageModel,
)

raw_sentences = [
    "the cat sat on the mat",
    "the cat sat on the rug",
    "the dog sat on the mat",
    "the dog ran to the park",
    "the cat ran to the park",
    "the dog sat on the rug",
]
training_data = tokenize(raw_sentences)
for sentence in training_data:
    print(sentence)

first_order = FirstOrderLanguageModel(training_data)
second_order = SecondOrderLanguageModel(training_data)

['<START>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '<END>']
['<START>', 'the', 'cat', 'sat', 'on', 'the', 'rug', '<END>']
['<START>', 'the', 'dog', 'sat', 'on', 'the', 'mat', '<END>']
['<START>', 'the', 'dog', 'ran', 'to', 'the', 'park', '<END>']
['<START>', 'the', 'cat', 'ran', 'to', 'the', 'park', '<END>']
['<START>', 'the', 'dog', 'sat', 'on', 'the', 'rug', '<END>']


## Part IV: Conditional probability tables

Each displayed row is a context; columns are possible next tokens. Empty cells correspond to zero-probability transitions.

In [2]:
selected_contexts = ["the", "cat", "dog", "sat", "ran"]
cpt_rows = []
for context in selected_contexts:
    distribution = first_order.distribution(context)
    for next_word, probability in distribution.items():
        cpt_rows.append({"current": context, "next": next_word, "probability": probability})
cpt = pd.DataFrame(cpt_rows).pivot(index="current", columns="next", values="probability").fillna(0)
display(cpt)

print("Examples of zero-probability transitions:")
for current, following in [("the", "sat"), ("cat", "dog"), ("dog", "cat"), ("sat", "mat"), ("ran", "sat")]:
    print(f"P({following} | {current}) = {first_order.distribution(current).get(following, 0.0)}")

next,cat,dog,mat,on,park,ran,rug,sat,to
current,,,,,,,,,
cat,0.00,0.00,0.000000,0.0,0.000000,0.333333,0.000000,0.666667,0.0
dog,0.00,0.00,0.000000,0.0,0.000000,0.333333,0.000000,0.666667,0.0
ran,0.00,0.00,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,1.0
sat,0.00,0.00,0.000000,1.0,0.000000,0.000000,0.000000,0.000000,0.0
the,0.25,0.25,0.166667,0.0,0.166667,0.000000,0.166667,0.000000,0.0


Examples of zero-probability transitions:
P(sat | the) = 0.0
P(dog | cat) = 0.0
P(cat | dog) = 0.0
P(mat | sat) = 0.0
P(sat | ran) = 0.0


## Parts V–VIII: Implementation inspection and testing

The behavioural specification is: count adjacent pairs; normalise each context's counts; display a specified conditional distribution; support greedy prediction and random sampling; stop at `<END>`; use no neural-network or pretrained language-model library.

In [3]:
# Probability normalisation invariant.
normalisation = pd.DataFrame([
    {"context": context, "sum_of_probabilities": sum(distribution.values())}
    for context, distribution in first_order.probabilities.items()
])
display(normalisation)
assert all(abs(total - 1.0) < 1e-12 for total in normalisation["sum_of_probabilities"])

# Next-word distributions and argmax predictions for five contexts.
for context in selected_contexts:
    distribution = first_order.distribution(context)
    print(f"P(next | {context}) = {distribution}")
    print(f"argmax = {first_order.predict(context)}\n")

,context,sum_of_probabilities
0,<START>,1.0
1,the,1.0
2,cat,1.0
3,sat,1.0
4,on,1.0
5,mat,1.0
6,rug,1.0
7,dog,1.0
8,ran,1.0
9,to,1.0


P(next | the) = {'cat': 0.25, 'mat': 0.16666666666666666, 'rug': 0.16666666666666666, 'dog': 0.25, 'park': 0.16666666666666666}
argmax = cat

P(next | cat) = {'sat': 0.6666666666666666, 'ran': 0.3333333333333333}
argmax = sat

P(next | dog) = {'sat': 0.6666666666666666, 'ran': 0.3333333333333333}
argmax = sat

P(next | sat) = {'on': 1.0}
argmax = on

P(next | ran) = {'to': 1.0}
argmax = to



## Part IX: Generate and save at least 20 sentences

In [4]:
rng = random.Random(42)
twenty_sentences = [first_order.generate(mode="sampling", rng=rng) for _ in range(20)]
for i, sentence in enumerate(twenty_sentences, 1):
    print(f"{i:2d}. {sentence}")

 1. the cat ran to the park
 2. the dog ran to the cat ran to the cat sat on the park
 3. the cat sat on the cat sat on the park
 4. the dog sat on the rug
 5. the cat ran to the cat ran to the dog sat on the rug
 6. the cat sat on the rug
 7. the park
 8. the cat ran to the cat sat on the mat
 9. the dog ran to the dog sat on the cat sat on the cat ran to the dog ran to the mat
10. the cat sat on the dog sat on the dog sat on the mat
11. the rug
12. the mat
13. the mat
14. the cat ran to the rug
15. the cat sat on the dog ran to the cat sat on the rug
16. the mat
17. the dog sat on the mat
18. the dog sat on the cat sat on the rug
19. the park
20. the rug


## Part X: Greedy versus probabilistic generation

In [5]:
greedy_five = [first_order.generate(mode="greedy") for _ in range(5)]
sampling_five = [first_order.generate(mode="sampling", rng=random.Random(100 + i)) for i in range(5)]
print("GREEDY:")
for sentence in greedy_five:
    print(sentence)
print("\nSAMPLING:")
for sentence in sampling_five:
    print(sentence)

GREEDY:
the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on the cat <TRUNCATED: NO END TOKEN>
the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on the cat <TRUNCATED: NO END TOKEN>
the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on the cat <TRUNCATED: NO END TOKEN>
the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on the cat <TRUNCATED: NO END TOKEN>
the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on the cat <TRUNCATED: NO END TOKEN>

SAMPLING:
the dog sat on the dog sat on the dog ran to the dog ran to the cat ran to the rug
the cat sat on the mat
the mat
the dog sat on the cat sat on the rug
the rug


## Parts XI–XIII: Second-order model and comparison

The second-order CPT is indexed by a pair \((X_{t-2},X_{t-1})\). Two `<START>` markers provide the initial context. No backoff is used, so an unseen pair has no defined distribution.

In [6]:
second_order_examples = [second_order.generate(mode="sampling", rng=random.Random(200 + i)) for i in range(20)]
for i, sentence in enumerate(second_order_examples, 1):
    print(f"{i:2d}. {sentence}")

# Compare both stored entries and independent free parameters.
non_end_tokens = sorted({t for s in training_data for t in s if t != END})
first_contexts = [(t,) for t in non_end_tokens]
second_contexts = list(product(non_end_tokens, repeat=2))
comparison = pd.DataFrame([
    {
        "model": "first-order",
        "observed_contexts": len(first_order.probabilities),
        "stored_positive_probabilities": sum(len(d) for d in first_order.probabilities.values()),
        "independent_free_parameters": sum(len(d)-1 for d in first_order.probabilities.values()),
        "zero_probability_contexts": sum(c not in first_order.probabilities for c in first_contexts),
        "unique_sentences_in_100_samples": len({first_order.generate(rng=random.Random(i)) for i in range(100)}),
    },
    {
        "model": "second-order",
        "observed_contexts": len(second_order.probabilities),
        "stored_positive_probabilities": sum(len(d) for d in second_order.probabilities.values()),
        "independent_free_parameters": sum(len(d)-1 for d in second_order.probabilities.values()),
        "zero_probability_contexts": sum(c not in second_order.probabilities for c in second_contexts),
        "unique_sentences_in_100_samples": len({second_order.generate(rng=random.Random(i)) for i in range(100)}),
    },
])
display(comparison)
print("First-order example :", first_order.generate(rng=random.Random(7)))
print("Second-order example:", second_order.generate(rng=random.Random(7)))

 1. the cat sat on the mat
 2. the cat ran to the park
 3. the dog sat on the rug
 4. the dog sat on the rug
 5. the cat sat on the mat
 6. the dog sat on the mat
 7. the cat sat on the rug
 8. the dog sat on the rug
 9. the dog ran to the park
10. the cat sat on the mat
11. the dog sat on the mat
12. the dog ran to the park
13. the dog sat on the mat
14. the cat ran to the park
15. the cat ran to the park
16. the dog ran to the park
17. the dog sat on the rug
18. the cat ran to the park
19. the cat sat on the rug
20. the cat ran to the park


,model,observed_contexts,stored_positive_probabilities,independent_free_parameters,zero_probability_contexts,unique_sentences_in_100_samples
0,first-order,11,17,6,11,36
1,second-order,15,19,4,106,6


First-order example : the cat sat on the dog ran to the dog ran to the park
Second-order example: the cat sat on the mat


In [7]:
# Required saved generated-text deliverable.
output_path = Path("generated_sentences.txt")
with output_path.open("w", encoding="utf-8") as f:
    f.write("FIRST-ORDER: 20 SAMPLED SENTENCES\n")
    f.write("\n".join(f"{i+1}. {s}" for i, s in enumerate(twenty_sentences)))
    f.write("\n\nFIRST-ORDER: FIVE GREEDY SENTENCES\n")
    f.write("\n".join(f"{i+1}. {s}" for i, s in enumerate(greedy_five)))
    f.write("\n\nFIRST-ORDER: FIVE NEW SAMPLED SENTENCES\n")
    f.write("\n".join(f"{i+1}. {s}" for i, s in enumerate(sampling_five)))
    f.write("\n\nSECOND-ORDER: 20 SAMPLED SENTENCES\n")
    f.write("\n".join(f"{i+1}. {s}" for i, s in enumerate(second_order_examples)))
print(f"Saved {len(twenty_sentences) + len(greedy_five) + len(sampling_five) + len(second_order_examples)} sentences to {output_path.resolve()}")

Saved 50 sentences to /mnt/agents/output/Solved_Labs/Bayesian_Networks_and_Autoregressive_Language_Models/generated_sentences.txt


# Answers to Questions 1–14

**1. Why is the chain-rule decomposition useful?** It expresses a whole-sequence probability as a product of next-token conditional probabilities. Generation then becomes a well-defined loop: sample a token from the current conditional distribution, append it, update the context, and repeat. It also turns training into learning conditional predictors rather than storing every complete sentence.

**2. What independence assumption does the first-order network make?** \(\boxed{P(X_t\mid X_1,\ldots,X_{t-1})=P(X_t\mid X_{t-1})}\). Equivalently, \(X_t\) is conditionally independent of all earlier tokens \(X_1,\ldots,X_{t-2}\) given \(X_{t-1}\).

**3. Conditional distributions and zeros.** The CPT above gives all nonzero probabilities. Explicitly, \(P(cat\mid the)=P(dog\mid the)=3/12=0.25\), and \(P(mat\mid the)=P(rug\mid the)=P(park\mid the)=2/12\approx0.1667\); \(P(sat\mid cat)=2/3\), \(P(ran\mid cat)=1/3\); \(P(sat\mid dog)=2/3\), \(P(ran\mid dog)=1/3\); \(P(on\mid sat)=1\); and \(P(to\mid ran)=1\). Unobserved pairs such as `the → sat`, `cat → dog`, `sat → mat`, and `ran → sat` have probability zero in this unsmoothed model.

**4. Where are transition counts stored?** In `FirstOrderLanguageModel.counts`, a `defaultdict(Counter)` mapping each previous token to counts of following tokens. In the second-order model, the key is a two-token context tuple.

**5. Where is \(P(X_t\mid X_{t-1})\) computed?** In the model constructor when `self.probabilities` divides each outcome count by `sum(outcomes.values())`. The `distribution()` method returns the resulting row.

**6. How is the next word chosen?** Both modes are implemented. `predict()` and greedy generation choose the argmax. Sampling generation uses `random.choices` with the conditional probabilities as weights. Greedy decoding repeats the same path for the same context; sampling can follow different paths while still respecting the learned distribution.

**7. What happens for an unseen word/context?** `distribution()` returns an empty dictionary. `predict()` and `generate()` then raise a descriptive `ValueError`; they do not silently invent probabilities. Smoothing or backoff would be alternative modelling choices, but are not part of this unsmoothed specification.

**8. What does a normalisation total of 0.87 mean?** The row is not a valid probability distribution: 0.13 of the probability mass is missing. Likely causes include omitted outcomes, incorrect normalisation, or accidental filtering. A successful run alone would not establish correctness.

**9. Do the argmax predictions always match human expectations?** Not necessarily. Some predictions, such as `sat → on` and `ran → to`, look natural; after `the`, the model can choose only `cat` in an argmax tie even though `dog` is equally probable and also natural. Humans use semantics, world knowledge, and broader context, whereas this model only reproduces local frequencies from six sentences.

**10. Which decoding mode produces more variation, and why?** Sampling produces more variation because it can select any token with nonzero probability. Greedy decoding always selects the same argmax for a given state, so all five greedy runs are identical. Sampling can produce different sentences across runs. Here greedy decoding also enters the deterministic cycle `the cat sat on the ...`, so the saved greedy outputs carry an explicit truncation marker rather than a fabricated `<END>`.

**11. How does second order differ?** (1) The graph has edges from both \(X_{t-2}\) and \(X_{t-1}\) into \(X_t\), rather than only the immediately previous token. (2) Its CPT has one row per ordered context pair and therefore many more possible rows. (3) Prediction uses two tokens of context. (4) It needs substantially more data because the same observations are divided among many more contexts, causing sparsity and zero-count pairs.

**12. Why can more context help but also hurt estimation?** Extra context can disambiguate what should follow, improving coherence. However, a vocabulary of size \(V\) has \(V\) possible first-order contexts but \(V^2\) possible second-order contexts, with up to \(V\) outcomes per row. Limited data must cover a much larger CPT, so many contexts are unseen or estimated from very few observations. The comparison table quantifies this sparsity and the resulting diversity difference; the printed examples show that the second-order model mostly reproduces training-like sentences while the first-order model can recombine fragments.

**13. Why is a precise behavioural specification preferable to “write a language model”?** It states the variables, dependencies, estimator, decoding rule, and stopping behaviour before implementation. Understanding the CPT representation lets us inspect counts and parent/context ordering. Normalisation tests and known-context predictions validate probabilistic invariants and semantics rather than merely syntax. Most importantly, code is an implementation of a model, not the model's definition: two implementations can run while representing different distributions.

**14. What did the Bayesian-network view add?** It made dependencies and independence assumptions explicit, supplied the factorisation used for generation, and gave each CPT row a precise conditional-probability interpretation. It also explained why increasing context enlarges the graph/CPT and provided a principled test target: an implementation must reproduce the specified factorisation and normalised conditionals, not just produce plausible text.

## Reflection on LLM use and validation

The LLM was given explicit model and behaviour specifications rather than a vague request. One candidate implementation used `max()` for every generation step; inspection showed that this implemented greedy decoding only, not the required sampling model. It was corrected by separating greedy and sampling modes and using probability-weighted `random.choices`. Explicit errors for unseen contexts were also added. Validation then checked every CPT row sums to 1, inspected known probabilities and argmax predictions, generated the required text, and compared parameter counts, zero-context counts, and sample diversity across both model orders.